# 08 Similarity Scale-Up

## Purpose
This notebook applies the similarity workflow to the full held-out test split instead of a small hand-picked probe set.

## Why this notebook matters
Notebook 7 is useful for controlled local examples, but this notebook answers the larger question: how does the selected model behave across the full held-out corpus, and what kinds of neighborhoods appear around a small set of query glycans?

## Inputs
- A saved tokenizer and model checkpoint from the project Drive workspace
- The held-out `test.txt` split stored in the Drive project folder
- A small user-defined panel of query glycans to inspect against the held-out corpus

## Outputs
- All-vs-all corpus similarity summaries and plots
- Specific-vs-all query summaries, threshold clouds, and HTML reports
- PCA visual summaries for selected query glycans
- A clean public-export folder that can be reviewed before any GitHub copy step


## Setup note

This notebook follows the same split-storage pattern as the rest of the project:
- notebook code lives in GitHub
- checkpoints, split files, and saved outputs live in Google Drive
- Colab pulls the repository at runtime so the notebook uses the current `src/` helper code

The held-out split used here is a plain-text file stored under `MyDrive/ProjectRoot/data/splits/`. Because the split file contains sequences rather than accession metadata, the notebook builds stable row identifiers for reporting after the file is loaded.


## Runtime setup

This cell prepares the Colab runtime for the notebook. It mounts Google Drive, synchronizes the GitHub repository, and makes the checked-out repository importable so the notebook can use the shared helpers in `src/`.

**Expected output**
- confirmation that Google Drive is mounted
- confirmation that the repository was cloned or updated
- the active repository directory inside the Colab runtime


In [ ]:
# Standard library imports used only for Colab runtime setup.
import os
import subprocess
import sys

from google.colab import drive

# Mount Google Drive before the notebook tries to read checkpoints, split files,
# or prior saved outputs from the shared project workspace.
drive.mount('/content/drive')

# Define the public repository that stores the notebooks and shared helpers.
GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
GITHUB_REF = 'main'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

# Clone the repository in a fresh runtime, or fast-forward the existing clone so
# the notebook uses the latest helper code.
if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)
else:
    print(f'Reusing existing repo at {REPO_DIR}')
    subprocess.run(
        ['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', GITHUB_REF],
        check=True,
    )

# Change into the repository so any notebook-side shell commands run from the
# same working directory as the project source tree.
%cd {REPO_DIR}

# Add the repository root to the Python import path so `src` imports resolve to
# the checked-out project code.
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Active repo directory: {REPO_DIR}')


## Import notebook dependencies

This cell imports the shared similarity helpers used throughout the notebook. The project modules are reloaded so a rerun in the same Colab session picks up fresh GitHub edits without requiring a full runtime restart.

**Expected output**
- no printed output under normal conditions
- a normal Python import error only if the repository sync step failed or a required dependency is missing


In [ ]:
# Import shared notebook dependencies after the repository is available locally.
import importlib
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

import src.glycan_cartoons as glycan_cartoons
import src.similarity as similarity
import src.similarity_core as similarity_core
import src.similarity_scaleup as similarity_scaleup
import src.similarity_variants as similarity_variants

# Reload implementation modules before the compatibility shim in src.similarity.
# That keeps the notebook from holding onto stale re-exported function objects
# during reruns in the same Colab session.
for module in (
    glycan_cartoons,
    similarity_core,
    similarity_scaleup,
    similarity_variants,
    similarity,
):
    importlib.reload(module)

from src.notebook_utils import SUPPORTED_TOKENIZER_FAMILIES, validate_tokenizer_family
from src.similarity import (
    build_active_cloud_preview,
    build_public_export_dir,
    build_public_report_subdir,
    build_ranked_neighbor_preview,
    build_tokenization_preview,
    export_public_scaleup_html,
    load_plaintext_sequence_corpus,
    load_similarity_artifacts,
    prepare_selected_query_panels,
    run_scaleup_similarity_analysis,
    save_scaleup_pca_outputs,
    validate_scaleup_similarity_inputs,
)


## User settings

This is the main cell to review before running the notebook. Keep edits here whenever possible so the rest of the notebook can remain stable and readable.

**Settings to review**
- `DRIVE_ROOT`: the root folder for the project in Google Drive
- `CHECKPOINT_SOURCE`, `TOKENIZER_FAMILY`, `EXPERIMENT_NAME`, and `CLASSIFIER_RUN_LABEL`: which saved model checkpoint to load
- `TEST_SPLIT_FILENAME`: which held-out split file to analyze
- `SELECTED_GLYCANS`, `RUN_QUERY_ACCESSIONS`, and `REVIEW_QUERY_ACCESSIONS`: which query glycans to analyze and display
- `SIMILARITY_THRESHOLDS`, `ACTIVE_CLOUD_THRESHOLD`, and PCA settings: how similarity neighborhoods are summarized
- `PUBLIC_EXPORT_ENABLED`, `PUBLIC_EXPORT_MODE`, and related settings: whether to package a clean shareable HTML folder and whether to export only the main landing page or the full report set

**Expected output**
- this cell only defines settings; it does not run the analysis


In [ ]:
from pathlib import Path

# Update DRIVE_ROOT if the project folder lives somewhere else in Google Drive.
DRIVE_ROOT = Path('/content/drive/MyDrive/ProjectRoot')
CHECKPOINTS_SUBDIR = 'checkpoints'
SPLITS_SUBDIR = 'data/splits'
RESULTS_SUBDIR = 'results/similarity_scaleup'

# Choose whether the checkpoint comes from pretraining or classification fine-tuning.
CHECKPOINT_SOURCE = 'classification'
TOKENIZER_FAMILY = 'manual'
EXPERIMENT_NAME = 'mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only'
CLASSIFIER_RUN_LABEL = 'cls_lr2e-5_ep100_bs16_mlm'
MODEL_SUBDIR = 'best_model'

# Select the held-out split file that should be used as the corpus.
TEST_SPLIT_FILENAME = 'test.txt'

# Control whether glycan cartoons are reused from cache or downloaded again.
CARTOON_LOOKUP_MODE = 'live'
CARTOON_DEVELOPER_EMAIL = ''
CARTOON_IMAGE_FORMAT = 'svg'
CARTOON_DISPLAY_MODE = 'extended'
LOOKUP_TIMEOUT = 60
CARTOON_CACHE_EXPERIMENT_NAME = 'mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only_v2_cont_lr5e-05_ep20'
CARTOON_CACHE_MANIFEST_FILENAME = 'scaleup_cartoon_manifest.csv'

# Define the query glycans that should be compared against the held-out corpus.
SELECTED_GLYCANS = [
    {
        'accession': 'G60230HH',
        'sequence': 'Mana1-2Mana1-2Mana1-3(Mana1-2Mana1-3(Mana1-2Mana1-6)Mana1-6)Manb1-4GlcNAcb1-4GlcNAcb',
        'label': 'High mannose N-glycan',
    },
    {
        'accession': 'G74120DW',
        'sequence': 'Galb1-4GlcNAcb1-2Mana1-3(Galb1-4GlcNAcb1-2(Galb1-4GlcNAcb1-4)Mana1-6)Manb1-4GlcNAcb1-4(Fuca1-6)GlcNAcb',
        'label': 'Complex N-glycan',
    },
    {
        'accession': 'G25140TA',
        'sequence': 'NeuAca2-6Galb1-4GlcNAcb1-2Mana1-3(GlcNAcb1-4)(Galb1-4GlcNAcb1-2Mana1-6)Manb1-4GlcNAcb1-4GlcNAcb',
        'label': 'Complex N-glycan w/ Sialic Acid',
    },
    {
        'accession': 'G27893KR',
        'sequence': 'Fuca1-2(GalNAca1-3)Galb1-3(Galb1-4GlcNAcb1-6)GalNAca',
        'label': 'O-glycan',
    },
]
RUN_QUERY_ACCESSIONS = ['G60230HH', 'G74120DW', 'G25140TA', 'G27893KR']
REVIEW_QUERY_ACCESSIONS = ['G60230HH', 'G74120DW', 'G25140TA', 'G27893KR']

# Configure the saved threshold clouds and notebook review settings.
SIMILARITY_THRESHOLDS = [0.95, 0.90, 0.85, 0.80]
ACTIVE_CLOUD_THRESHOLD = 0.90
ALL_VS_ALL_TOP_K = 10
HTML_NEIGHBOR_LIMIT = 50
HTML_CLOUD_LIMIT = 100
NOTEBOOK_NEIGHBOR_LIMIT = 15
NOTEBOOK_CLOUD_DISPLAY_LIMIT = 25

# Configure tokenization and PCA behavior.
MAX_LENGTH = None
BATCH_SIZE = 32
POOLING_STRATEGY = 'mean'
PCA_FOCUS_ACCESSIONS = None
PCA_BACKGROUND_SAMPLE_SIZE = 2000
PCA_RANDOM_STATE = 7
PCA_BACKGROUND_POINT_SIZE = 10
PCA_CLOUD_POINT_SIZE = 28
PCA_QUERY_POINT_SIZE = 110

# Configure the optional clean public-export folder.
PUBLIC_EXPORT_ENABLED = True
# Use 'index_only' for a faster export that keeps the main landing page and its direct assets.
# Switch to 'full_report_set' only when you also need every accession-specific HTML page copied.
PUBLIC_EXPORT_MODE = 'index_only'
PUBLIC_EXPORT_PARENT_SUBDIR = 'results/public_reports'
REPORT_TITLE = 'Test-Set Similarity Report'
PUBLIC_GITHUB_OWNER = 'hb791-dev'
PUBLIC_GITHUB_REPO = 'glycan-roberta'
PUBLIC_GITHUB_REF = 'main'
PUBLIC_EXPORT_FAIL_ON_SENSITIVE_MATCH = True


## Resolve notebook paths and run metadata

This cell converts the editable settings into the concrete paths and labels used by the workflow. It also validates the main categorical settings before the model is loaded.

**Expected output**
- the resolved model directory and output directory
- the test split path
- the planned report title and public-export target

**How to interpret the result**
- if the model directory looks wrong, review the checkpoint settings in the user-settings cell
- if the public-export path looks wrong, review the report-label settings before running the analysis


In [ ]:
# Validate categorical settings before building derived paths.
validate_tokenizer_family(TOKENIZER_FAMILY, supported_families=SUPPORTED_TOKENIZER_FAMILIES)

if CHECKPOINT_SOURCE not in {'pretraining', 'classification'}:
    raise ValueError("CHECKPOINT_SOURCE must be either 'pretraining' or 'classification'.")

if POOLING_STRATEGY not in {'cls', 'mean', 'max'}:
    raise ValueError("POOLING_STRATEGY must be one of 'cls', 'mean', or 'max'.")

if CARTOON_LOOKUP_MODE not in {'cache_only', 'live'}:
    raise ValueError("CARTOON_LOOKUP_MODE must be either 'cache_only' or 'live'.")

if PUBLIC_EXPORT_MODE not in {'index_only', 'full_report_set'}:
    raise ValueError("PUBLIC_EXPORT_MODE must be either 'index_only' or 'full_report_set'.")

# Build the shared project paths used by the analysis.
CHECKPOINTS_DIR = DRIVE_ROOT / CHECKPOINTS_SUBDIR
SPLITS_DIR = DRIVE_ROOT / SPLITS_SUBDIR
SCALEUP_RESULTS_DIR = DRIVE_ROOT / RESULTS_SUBDIR
TEST_SPLIT_PATH = SPLITS_DIR / TEST_SPLIT_FILENAME

# Build the output labels from the selected checkpoint and pooling strategy.
POOLING_LABEL = f'{POOLING_STRATEGY}_pool'
BASE_OUTPUT_RUN_LABEL = (
    'cache_only_example' if CARTOON_LOOKUP_MODE == 'cache_only'
    else f'live_{CARTOON_DISPLAY_MODE}'
)
MODEL_OUTPUT_ID = 'pretrained_mlm' if CHECKPOINT_SOURCE == 'pretraining' else CLASSIFIER_RUN_LABEL
OUTPUT_LABEL = f'{TOKENIZER_FAMILY}__{MODEL_OUTPUT_ID}__{POOLING_LABEL}'
OUTPUT_RUN_LABEL = f'{BASE_OUTPUT_RUN_LABEL}__{OUTPUT_LABEL}'
OUTPUT_NAME = f'{REPORT_TITLE} | {TOKENIZER_FAMILY} | {MODEL_OUTPUT_ID} | {POOLING_LABEL}'

# Resolve the checkpoint path differently for pretrained and classification runs.
if CHECKPOINT_SOURCE == 'classification':
    if not str(CLASSIFIER_RUN_LABEL).strip():
        raise ValueError('Set CLASSIFIER_RUN_LABEL when CHECKPOINT_SOURCE = classification.')

    MODEL_DIR = (
        CHECKPOINTS_DIR
        / 'classification'
        / TOKENIZER_FAMILY
        / EXPERIMENT_NAME
        / CLASSIFIER_RUN_LABEL
        / MODEL_SUBDIR
    )
    MODEL_PATH_PARTS = ['classification', TOKENIZER_FAMILY, EXPERIMENT_NAME, CLASSIFIER_RUN_LABEL]
    REPORT_SUBTITLE = (
        f'Model run: {TOKENIZER_FAMILY} tokenizer | {EXPERIMENT_NAME} | '
        f'{CLASSIFIER_RUN_LABEL} (classification fine-tuned) | pooling={POOLING_STRATEGY}'
    )
else:
    MODEL_DIR = CHECKPOINTS_DIR / TOKENIZER_FAMILY / EXPERIMENT_NAME / MODEL_SUBDIR
    MODEL_PATH_PARTS = [TOKENIZER_FAMILY, EXPERIMENT_NAME]
    REPORT_SUBTITLE = (
        f'Model run: {TOKENIZER_FAMILY} tokenizer | {EXPERIMENT_NAME} | '
        f'pretrained_mlm | pooling={POOLING_STRATEGY}'
    )

OUTPUT_DIR = SCALEUP_RESULTS_DIR.joinpath(*MODEL_PATH_PARTS, OUTPUT_RUN_LABEL)
PUBLIC_REPORT_NOTEBOOK_STEM = '08_similarity_scaleup'
PUBLIC_EXPORT_PATH_PARTS = [TOKENIZER_FAMILY, EXPERIMENT_NAME, MODEL_OUTPUT_ID, OUTPUT_RUN_LABEL]
PUBLIC_EXPORT_PARENT_DIR = DRIVE_ROOT / PUBLIC_EXPORT_PARENT_SUBDIR
PUBLIC_EXPORT_REPO_SUBDIR = build_public_report_subdir(
    PUBLIC_REPORT_NOTEBOOK_STEM,
    PUBLIC_EXPORT_PATH_PARTS,
)
PUBLIC_EXPORT_DIR = build_public_export_dir(
    PUBLIC_EXPORT_PARENT_DIR,
    PUBLIC_REPORT_NOTEBOOK_STEM,
    PUBLIC_EXPORT_PATH_PARTS,
)
PUBLIC_EXPORT_EXTRA_BLOCKED_STRINGS = [
    value for value in [CARTOON_DEVELOPER_EMAIL]
    if str(value).strip()
]
CARTOON_CACHE_MANIFEST_PATH = (
    SCALEUP_RESULTS_DIR
    / TOKENIZER_FAMILY
    / CARTOON_CACHE_EXPERIMENT_NAME
    / CARTOON_CACHE_MANIFEST_FILENAME
)

if CARTOON_LOOKUP_MODE == 'cache_only' and not CARTOON_CACHE_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'Cartoon cache manifest not found: {CARTOON_CACHE_MANIFEST_PATH}'
    )

# Create the top-level results folder early so path problems appear before the
# more expensive embedding work begins.
SCALEUP_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Drive root: {DRIVE_ROOT}')
print(f'Model directory: {MODEL_DIR}')
print(f'Test split path: {TEST_SPLIT_PATH}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Report title: {OUTPUT_NAME}')
print(f'Report subtitle: {REPORT_SUBTITLE}')
print(f'Public export enabled: {PUBLIC_EXPORT_ENABLED}')
print(f'Public export mode: {PUBLIC_EXPORT_MODE}')
print(f'Public export Drive folder: {PUBLIC_EXPORT_DIR}')
print(f'Repo destination after copy: {PUBLIC_EXPORT_REPO_SUBDIR}')


## Load the held-out corpus

This cell reads the plain-text held-out split and converts it into a dataframe with stable accession-style row identifiers. Those identifiers make the saved CSVs and HTML reports easier to interpret than a sequence-only table.

**Expected output**
- the total number of held-out glycans
- a short preview table showing the generated row identifiers and sequences

**How to interpret the result**
- if the count is unexpectedly small, review `TEST_SPLIT_FILENAME` and `DRIVE_ROOT`
- if the preview looks malformed, stop here before running the model


In [ ]:
# Load the full held-out corpus once so every later analysis step uses the same rows.
test_glycans_df = load_plaintext_sequence_corpus(TEST_SPLIT_PATH)

print(f'Held-out test glycans: {len(test_glycans_df):,}')
display(test_glycans_df.head(10))


## Build the query panels from the user settings

This cell converts the selected-glycan settings into the exact query panel used for the run, the smaller review panel used in the notebook display, and the effective PCA focus list.

**Expected output**
- a table of the query glycans that will be analyzed
- the ordered run panel and review panel accessions
- the resolved PCA focus accessions

**How to interpret the result**
- if a requested review accession is missing, review `RUN_QUERY_ACCESSIONS` and `REVIEW_QUERY_ACCESSIONS`
- if the PCA focus list is unexpected, review `PCA_FOCUS_ACCESSIONS`


In [ ]:
# Validate the selected-glycan settings and organize the run and review panels.
query_panel_artifacts = prepare_selected_query_panels(
    selected_glycans=SELECTED_GLYCANS,
    run_query_accessions=RUN_QUERY_ACCESSIONS,
    review_query_accessions=REVIEW_QUERY_ACCESSIONS,
    pca_focus_accessions=PCA_FOCUS_ACCESSIONS,
    active_cloud_threshold=ACTIVE_CLOUD_THRESHOLD,
    similarity_thresholds=SIMILARITY_THRESHOLDS,
)

selected_glycans_df = query_panel_artifacts['selected_glycans_df']
selected_accessions = query_panel_artifacts['selected_accessions']
review_accessions = query_panel_artifacts['review_accessions']
effective_pca_focus_accessions = query_panel_artifacts['effective_pca_focus_accessions']
selected_glycan_lookup = query_panel_artifacts['selected_glycan_lookup']

display(selected_glycans_df[['accession', 'label', 'sequence']])
print(f'Run panel: {selected_accessions}')
print(f'Review panel: {review_accessions}')
print(f'Active notebook cloud threshold: {ACTIVE_CLOUD_THRESHOLD:.2f}')
print(f'PCA focus accessions: {effective_pca_focus_accessions}')


## Validate inputs, load the model, and run the analysis

This is the main workflow cell. It validates the notebook inputs, loads the tokenizer and checkpoint, builds the full held-out similarity outputs, and summarizes cartoon-download status.

**Expected output**
- a tokenization preview for the selected glycans
- cartoon manifest summary tables
- a short report of any failed cartoon lookups

**How to interpret the result**
- if validation fails immediately, review the checkpoint path or selected-glycan settings
- if the tokenization preview looks wrong, stop before interpreting the similarity outputs
- if cartoon failures appear, the similarity tables are still usable, but the HTML reports may have missing images


In [ ]:
# Validate the main inputs before the notebook loads the model checkpoint.
validate_scaleup_similarity_inputs(
    model_dir=MODEL_DIR,
    corpus_df=test_glycans_df[['accession', 'sequence']],
    query_df=selected_glycans_df[['accession', 'sequence']],
    accession_col='accession',
    sequence_col='sequence',
    output_dir=OUTPUT_DIR,
)

# Load the tokenizer and encoder checkpoint used for the similarity analysis.
tokenizer, model, device = load_similarity_artifacts(str(MODEL_DIR))

# Preview how the selected glycans are tokenized before the notebook starts the
# more expensive embedding work.
selected_tokenization_preview_df = build_tokenization_preview(
    selected_glycans_df['sequence'].tolist(),
    tokenizer=tokenizer,
)

# In cache-only mode, the notebook reuses only existing manifest rows and local
# image files instead of attempting any new cartoon downloads.
cartoon_cache_manifest_for_run = (
    CARTOON_CACHE_MANIFEST_PATH if CARTOON_LOOKUP_MODE == 'cache_only' else None
)
cartoon_cache_only_for_run = CARTOON_LOOKUP_MODE == 'cache_only'

# Run the full scale-up workflow and save the resulting tables, plots, and HTML files.
results = run_scaleup_similarity_analysis(
    tokenizer=tokenizer,
    model=model,
    corpus_df=test_glycans_df[['accession', 'sequence']],
    query_df=selected_glycans_df[['accession', 'sequence']],
    output_dir=OUTPUT_DIR,
    output_name=OUTPUT_NAME,
    output_subtitle=REPORT_SUBTITLE,
    developer_email=CARTOON_DEVELOPER_EMAIL,
    accession_col='accession',
    sequence_col='sequence',
    thresholds=SIMILARITY_THRESHOLDS,
    cartoon_image_format=CARTOON_IMAGE_FORMAT,
    cartoon_display_mode=CARTOON_DISPLAY_MODE,
    lookup_timeout=LOOKUP_TIMEOUT,
    device=device,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    pooling_strategy=POOLING_STRATEGY,
    all_vs_all_top_k=ALL_VS_ALL_TOP_K,
    html_neighbor_limit=HTML_NEIGHBOR_LIMIT,
    html_cloud_limit=HTML_CLOUD_LIMIT,
    model_dir=MODEL_DIR,
    cartoon_cache_manifest_path=cartoon_cache_manifest_for_run,
    cartoon_cache_only=cartoon_cache_only_for_run,
)

# Load the saved cartoon manifest immediately so the notebook can summarize the
# image layer alongside the similarity outputs.
cartoon_manifest_df = pd.read_csv(results['saved_paths']['cartoon_manifest_path'])
cartoon_summary_tables = glycan_cartoons.summarize_cartoon_manifest(cartoon_manifest_df)
results['cartoon_manifest_df'] = cartoon_manifest_df
results['cartoon_summary_tables'] = cartoon_summary_tables

print('=== Selected glycan tokenization preview ===')
display(selected_tokenization_preview_df)
print('=== Cartoon fetch summary ===')
display(cartoon_summary_tables['overview_df'])
print('Lookup status breakdown:')
display(cartoon_summary_tables['lookup_status_df'])
print('Local image status breakdown:')
display(cartoon_summary_tables['local_image_status_df'])

failed_cartoons_df = cartoon_manifest_df.loc[
    cartoon_manifest_df['lookup_status'].isin(['lookup_error', 'parse_error', 'cache_only_miss'])
].copy()

print('Failed cartoon lookups:')
if failed_cartoons_df.empty:
    print('No lookup_error, parse_error, or cache_only_miss rows were found in this run.')
else:
    display(
        failed_cartoons_df[
            [
                'sequence',
                'lookup_status',
                'lookup_errors',
                'local_image_status',
                'image_url',
            ]
        ].head(25)
    )


## Review the all-vs-all corpus summary

This cell provides the broadest view of the embedding space by summarizing similarity values across the full held-out corpus before any query-specific interpretation.

**Expected output**
- a one-row summary table for the full non-self corpus distribution
- a preview of the nearest-neighbor pairs in the corpus
- the saved all-vs-all similarity histogram

**How to interpret the result**
- the summary table provides the baseline distribution that the query-specific results should be compared against
- the neighbor preview is a practical sanity check for obviously implausible close pairs


In [ ]:
print('=== All-vs-all summary ===')
display(results['all_vs_all_artifacts']['off_diagonal_summary_df'])

print('=== All-vs-all top-neighbor preview ===')
display(results['all_vs_all_artifacts']['top_neighbors_df'].head(25))

print('=== All-vs-all histogram ===')
display(Image(filename=str(results['saved_paths']['all_vs_all_histogram_path'])))


## Review the specific-vs-all query outputs

This cell shows the main per-query outputs used for interpretation: score summaries, threshold clouds, top-neighbor previews, and the saved histogram for each reviewed query glycan.

**Expected output**
- one set of summary tables and plots for each accession in the review panel
- a reminder of the accession-specific HTML report path for later browsing

**How to interpret the result**
- the distribution summary describes how each query relates to the full held-out corpus
- the top-neighbor table emphasizes rank order, while the threshold cloud emphasizes membership at a specific cutoff
- the histogram provides a quick visual check for whether the cloud threshold is too strict or too loose


In [ ]:
print('=== Notebook review controls ===')
print(f'- Review accessions: {review_accessions}')
print(f'- Active cloud threshold: {ACTIVE_CLOUD_THRESHOLD:.2f}')
print(f'- Ranked-neighbor rows shown: {NOTEBOOK_NEIGHBOR_LIMIT}')
print(f'- Cloud rows shown: {NOTEBOOK_CLOUD_DISPLAY_LIMIT}')

for accession in review_accessions:
    query_label = selected_glycan_lookup[accession]['label']
    ranked_neighbors_df = build_ranked_neighbor_preview(
        results_bundle=results,
        accession=accession,
        neighbor_limit=NOTEBOOK_NEIGHBOR_LIMIT,
    )
    active_cloud_df = build_active_cloud_preview(
        results_bundle=results,
        accession=accession,
        threshold=ACTIVE_CLOUD_THRESHOLD,
        cloud_limit=NOTEBOOK_CLOUD_DISPLAY_LIMIT,
    )

    print(f'=== {accession} ({query_label}) distribution summary ===')
    display(
        results['specific_vs_all_summary_df'].loc[
            results['specific_vs_all_summary_df']['query_accession'] == accession
        ]
    )

    print(f'=== {accession} threshold summary across all saved cutoffs ===')
    display(
        results['threshold_summary_df'].loc[
            results['threshold_summary_df']['query_accession'] == accession
        ]
    )

    print(f'=== {accession} top neighbors ===')
    display(ranked_neighbors_df)

    print(f'=== {accession} active cloud at threshold >= {ACTIVE_CLOUD_THRESHOLD:.2f} ===')
    if active_cloud_df.empty:
        print('No non-self held-out glycans cleared the current notebook cloud threshold.')
    else:
        display(active_cloud_df)

    print(f'=== {accession} histogram ===')
    display(Image(filename=str(results['saved_paths']['query_histogram_paths'][accession])))
    print(f'HTML report: {results["saved_paths"]["query_html_paths"][accession]}')


## Review the PCA embedding view

This cell creates one or more PCA projections from the already-computed embeddings so the selected queries can be viewed relative to the held-out background and the active similarity cloud.

**Expected output**
- one PCA image per focus accession
- a table of the selected-glycan PCA coordinates
- the explained variance for the two displayed principal components

**How to interpret the result**
- PCA is a supporting visualization rather than the primary evidence for similarity quality
- the figure is most useful for checking whether a query sits inside a dense neighborhood, on the edge of the space, or far from most background points


In [ ]:
# Save one or more PCA views from the embeddings that were already computed above.
pca_artifacts = save_scaleup_pca_outputs(
    results_bundle=results,
    query_metadata_df=selected_glycans_df[['accession', 'label', 'sequence']],
    output_dir=OUTPUT_DIR,
    focus_accessions=effective_pca_focus_accessions,
    threshold=ACTIVE_CLOUD_THRESHOLD,
    background_sample_size=PCA_BACKGROUND_SAMPLE_SIZE,
    random_state=PCA_RANDOM_STATE,
    background_point_size=PCA_BACKGROUND_POINT_SIZE,
    cloud_point_size=PCA_CLOUD_POINT_SIZE,
    query_point_size=PCA_QUERY_POINT_SIZE,
)

# Add the PCA files to the saved-path summary printed at the end of the notebook.
results['saved_paths'].update(
    {
        'pca_image_path': pca_artifacts['pca_image_path'],
        'pca_image_paths': pca_artifacts['pca_image_paths'],
        'pca_html_paths': pca_artifacts['pca_html_paths'],
        'pca_coordinates_path': pca_artifacts['pca_coordinates_path'],
        'pca_selected_path': pca_artifacts['pca_selected_path'],
    }
)

print(f'PCA focus accessions: {pca_artifacts["focus_accessions"]}')
print(f'Active cloud threshold: {pca_artifacts["threshold"]:.2f}')
print(
    'Explained variance by PC1 and PC2: '
    f'{pca_artifacts["explained_variance"][0]:.1f}% + {pca_artifacts["explained_variance"][1]:.1f}%'
)
print(f'Saved PCA coordinates: {pca_artifacts["pca_coordinates_path"]}')

for focus_accession in pca_artifacts['focus_accessions']:
    print(f'=== PCA view for {focus_accession} ===')
    print(f'Cloud size: {pca_artifacts["focus_cloud_sizes"][focus_accession]}')
    print(f'Background points plotted: {pca_artifacts["background_counts"][focus_accession]}')
    print(f'Saved PCA image: {pca_artifacts["pca_image_paths"][focus_accession]}')
    if focus_accession in pca_artifacts['pca_html_paths']:
        print(f'Updated HTML report: {pca_artifacts["pca_html_paths"][focus_accession]}')
    display(Image(filename=str(pca_artifacts['pca_image_paths'][focus_accession])))

display(pca_artifacts['selected_coordinates_df'])


## Prepare the clean public HTML export

This cell copies the browser-facing report files into a smaller public-export folder so the report can be reviewed before any manual copy into the GitHub repository. By default, the notebook uses a faster `index_only` export that keeps the main landing page and the direct assets it needs. Switch `PUBLIC_EXPORT_MODE` to `full_report_set` only when you also need every accession-specific HTML page copied into the public folder.

**Expected output**
- the clean export directory
- a table of copied files
- dependency and sensitive-string scan tables

**How to interpret the result**
- dependency issues mean the exported folder would break in a browser and should be fixed before any copy step
- sensitive-string matches should be reviewed before sharing the exported files publicly


In [ ]:
public_export_artifacts = None

if PUBLIC_EXPORT_ENABLED:
    public_export_artifacts = export_public_scaleup_html(
        results_bundle=results,
        export_dir=PUBLIC_EXPORT_DIR,
        repo_public_subdir=PUBLIC_EXPORT_REPO_SUBDIR,
        repo_owner=PUBLIC_GITHUB_OWNER,
        repo_name=PUBLIC_GITHUB_REPO,
        repo_ref=PUBLIC_GITHUB_REF,
        extra_blocked_strings=PUBLIC_EXPORT_EXTRA_BLOCKED_STRINGS,
        export_mode=PUBLIC_EXPORT_MODE,
    )

    results['saved_paths'].update(
        {
            'public_export_dir': public_export_artifacts['public_export_dir'],
            'public_export_repo_path': public_export_artifacts['repo_index_path'],
            'public_export_githack_url': public_export_artifacts['githack_url'],
        }
    )

    print(f'Public export mode: {public_export_artifacts["export_mode"]}')
    print(f'Public export Drive folder: {public_export_artifacts["public_export_dir"]}')
    print(f'Repo folder to copy into before push: {PUBLIC_EXPORT_REPO_SUBDIR}')
    print(f'Repo index path after push: {public_export_artifacts["repo_index_path"]}')
    print(f'GitHack URL after push: {public_export_artifacts["githack_url"]}')
    print('')

    print('=== Copied public files ===')
    display(public_export_artifacts['copied_files_df'])

    print('=== Dependency issues ===')
    if public_export_artifacts['dependency_issues_df'].empty:
        print('No missing local HTML dependencies were found in the public export.')
    else:
        display(public_export_artifacts['dependency_issues_df'])

    print('=== Sensitive-string scan ===')
    if public_export_artifacts['scan_results_df'].empty:
        print('No obvious personal paths or email strings were found in the copied files.')
    else:
        display(public_export_artifacts['scan_results_df'])

    if public_export_artifacts['has_dependency_issues']:
        raise ValueError(
            'The public export still has missing local dependencies. Fix those before any GitHub copy step.'
        )

    if public_export_artifacts['has_sensitive_matches'] and PUBLIC_EXPORT_FAIL_ON_SENSITIVE_MATCH:
        raise ValueError(
            'The public export still contains suspicious personal or local-environment strings. Review the scan table before sharing the files.'
        )
else:
    print('PUBLIC_EXPORT_ENABLED is False, so the notebook skipped the clean public-export step.')


## Saved outputs

This final cell prints every saved artifact path so the CSVs, plots, HTML reports, and optional public-export files can be located quickly after the run finishes.

**Expected output**
- one path listing for every saved artifact collected during the notebook run


In [ ]:
print('Saved outputs:')
for label, path in results['saved_paths'].items():
    if isinstance(path, dict):
        print(f'- {label}:')
        for child_label, child_path in path.items():
            print(f'    - {child_label}: {child_path}')
    else:
        print(f'- {label}: {path}')
